In [12]:
!pip install ninja

In [13]:
!cd kernelforge/ && git pull origin main


From https://github.com/marcalph/kernelforge
 * branch            main       -> FETCH_HEAD
Already up to date.


In [14]:
# Run this cell if load_inline fails with "cannot open shared object file"
# (happens when a previous compilation failed and left a broken cache entry)
!rm -rf /root/.cache/torch_extensions/py312_cu128/rgb_to_grayscale_extension/


In [ ]:
from pathlib import Path
import torch
from torchvision.io import read_image, write_png
from torch.utils.cpp_extension import load_inline


def compile_extension():
    cuda_source = Path("kernelforge/src/kernels/grayscale.cu").read_text()
    cpp_source = "torch::Tensor rgb_to_grayscale(torch::Tensor image);"

    return load_inline(
        name="rgb_to_grayscale_extension",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=["rgb_to_grayscale"],
        with_cuda=True,
        extra_cuda_cflags=["-O2"],
        verbose=True,
    )


def main():
    """
    Read input image, convert it to grayscale via custom CUDA kernel and write out as png.
    """
    ext = compile_extension()

    x = read_image("kernelforge/src/lenna.jpg").permute(1, 2, 0).cuda()
    print("Input image:", x.shape, x.dtype, "mean:", x.float().mean().item())

    assert x.dtype == torch.uint8

    y = ext.rgb_to_grayscale(x)

    print("Output image:", y.shape, y.dtype, "mean:", y.float().mean().item())
    write_png(y.permute(2, 0, 1).cpu(), "output.png")


main()
